# Advanced Problems: Modifying and Extending `namedtuple`

This notebook contains advanced, solution-driven exercises on immutable `namedtuple` instances, `_replace`, `_make`, `_fields`, validation patterns, safe extension, and conversion workflows.

## Learning goals

By the end, you should be able to:

- Update immutable named tuple records without mutation.
- Use `_replace` correctly and avoid brittle positional reconstruction.
- Extend named tuple schemas using `_fields`.
- Convert older records into newer schemas.
- Build reusable utilities around named tuples.
- Preserve immutability while implementing data-cleaning and transformation pipelines.


In [1]:
from collections import namedtuple
from statistics import mean

## Problem 1 — Safe multi-field update without mutation

You are given a named tuple representing daily market data.

Create a function `normalize_stock(record)` that returns a new record with:

1. `symbol` uppercased.
2. `low`, `high`, `open`, and `close` converted to floats.
3. If `low > high`, swap them.
4. The original object must not be modified.

Use `_replace`; do not reconstruct the object with positional arguments.


In [2]:
Stock = namedtuple('Stock', 'symbol year month day open high low close')

raw = Stock('djia', 2018, 1, 25, '26313', '26260', '26458', '26393')
raw

Stock(symbol='djia', year=2018, month=1, day=25, open='26313', high='26260', low='26458', close='26393')

### Solution

In [3]:
def normalize_stock(record):
    open_ = float(record.open)
    high = float(record.high)
    low = float(record.low)
    close = float(record.close)

    if low > high:
        low, high = high, low

    return record._replace(
        symbol=record.symbol.upper(),
        open=open_,
        high=high,
        low=low,
        close=close
    )

normalized = normalize_stock(raw)

print('Original:', raw)
print('Normalized:', normalized)
print('Same object?', raw is normalized)

assert raw.symbol == 'djia'
assert normalized.symbol == 'DJIA'
assert normalized.low <= normalized.high
assert isinstance(normalized.close, float)

Original: Stock(symbol='djia', year=2018, month=1, day=25, open='26313', high='26260', low='26458', close='26393')
Normalized: Stock(symbol='DJIA', year=2018, month=1, day=25, open=26313.0, high=26458.0, low=26260.0, close=26393.0)
Same object? False


### Why this is the best approach

`namedtuple` instances are immutable, so a transformation should create a new value instead of mutating the old one. `_replace` is preferred here because it is explicit, readable, and robust if field order is long or easy to confuse.


## Problem 2 — Generic validated replacement utility

Write a function `strict_replace(record, **changes)` that behaves like `_replace`, but adds two validations:

1. Every key in `changes` must be a valid field name.
2. No changed value may be `None`.

The function should return a new instance of the same named tuple type.


In [4]:
Trade = namedtuple('Trade', 'symbol quantity price side')

trade = Trade('AAPL', 100, 189.50, 'buy')
trade

Trade(symbol='AAPL', quantity=100, price=189.5, side='buy')

### Solution

In [5]:
def strict_replace(record, **changes):
    valid_fields = set(record._fields)
    unknown = set(changes) - valid_fields

    if unknown:
        raise ValueError(f'Unknown field(s): {sorted(unknown)}')

    none_fields = [field for field, value in changes.items() if value is None]
    if none_fields:
        raise ValueError(f'None is not allowed for field(s): {none_fields}')

    return record._replace(**changes)

updated_trade = strict_replace(trade, quantity=150, price=190.25)

print(updated_trade)
print(type(updated_trade) is type(trade))

assert updated_trade == Trade('AAPL', 150, 190.25, 'buy')
assert trade == Trade('AAPL', 100, 189.50, 'buy')

Trade(symbol='AAPL', quantity=150, price=190.25, side='buy')
True


In [6]:
# Uncomment each line to test the validation behavior.

# strict_replace(trade, qty=200)
# strict_replace(trade, price=None)

### Key idea

Although `_replace` already rejects invalid field names, wrapping it lets you enforce domain-specific rules before creating the updated record.


## Problem 3 — Extend an existing schema safely

You have an old `Stock` named tuple with eight fields.

Create a new named tuple type called `StockWithMetrics` that contains all original `Stock` fields plus:

- `previous_close`
- `change`
- `change_percent`

Then write `add_metrics(stock, previous_close)` that returns a `StockWithMetrics` instance.

Use `_fields` to build the extended schema.


In [7]:
Stock = namedtuple('Stock', 'symbol year month day open high low close')

stock = Stock('MSFT', 2024, 5, 15, 420.10, 425.00, 418.50, 423.25)
stock

Stock(symbol='MSFT', year=2024, month=5, day=15, open=420.1, high=425.0, low=418.5, close=423.25)

### Solution

In [8]:
StockWithMetrics = namedtuple(
    'StockWithMetrics',
    Stock._fields + ('previous_close', 'change', 'change_percent')
)

def add_metrics(stock, previous_close):
    change = stock.close - previous_close
    change_percent = change / previous_close * 100

    return StockWithMetrics._make(
        stock + (previous_close, change, change_percent)
    )

stock_metrics = add_metrics(stock, previous_close=419.00)

stock_metrics

StockWithMetrics(symbol='MSFT', year=2024, month=5, day=15, open=420.1, high=425.0, low=418.5, close=423.25, previous_close=419.0, change=4.25, change_percent=1.0143198090692125)

In [9]:
assert stock_metrics.symbol == stock.symbol
assert stock_metrics.previous_close == 419.00
assert round(stock_metrics.change, 2) == 4.25
assert round(stock_metrics.change_percent, 4) == round(4.25 / 419.00 * 100, 4)
assert StockWithMetrics._fields == Stock._fields + ('previous_close', 'change', 'change_percent')

### Best-practice note

Using `_fields` avoids duplicating the original schema manually. This reduces maintenance risk when the base named tuple changes.


## Problem 4 — Convert records between compatible named tuple versions

A data pipeline has two versions of a record:

- `QuoteV1`: `symbol bid ask`
- `QuoteV2`: `symbol bid ask mid spread`

Write `upgrade_quote(quote)` that converts a `QuoteV1` record into `QuoteV2`.

Rules:

1. `mid` is the average of `bid` and `ask`.
2. `spread` is `ask - bid`.
3. Reject records where `bid > ask`.
4. Use unpacking or `_make`, not repeated positional field access.


In [10]:
QuoteV1 = namedtuple('QuoteV1', 'symbol bid ask')
QuoteV2 = namedtuple('QuoteV2', QuoteV1._fields + ('mid', 'spread'))

quote = QuoteV1('EURUSD', 1.0842, 1.0845)
quote

QuoteV1(symbol='EURUSD', bid=1.0842, ask=1.0845)

### Solution

In [11]:
def upgrade_quote(quote):
    if quote.bid > quote.ask:
        raise ValueError('Invalid quote: bid cannot be greater than ask.')

    mid = mean((quote.bid, quote.ask))
    spread = quote.ask - quote.bid

    return QuoteV2._make(quote + (mid, spread))

quote_v2 = upgrade_quote(quote)
quote_v2

QuoteV2(symbol='EURUSD', bid=1.0842, ask=1.0845, mid=1.0843500000000001, spread=0.00029999999999996696)

In [12]:
assert quote_v2 == QuoteV2('EURUSD', 1.0842, 1.0845, 1.08435, 0.00029999999999996696)

try:
    upgrade_quote(QuoteV1('BAD', 10, 9))
except ValueError as ex:
    print(type(ex).__name__, ex)

AssertionError: 

### Design insight

This keeps the conversion logic centralized. If many records must be upgraded, the same function can be mapped across a list, generator, or file import pipeline.


## Problem 5 — Bulk transformation pipeline

You are given a list of immutable `Employee` records.

Create a function `promote(records, department, raise_percent)` that:

1. Finds employees in the target department.
2. Returns a new list where only those employees have their salary increased.
3. Does not mutate or replace unaffected employees unnecessarily.
4. Rounds updated salaries to two decimal places.

Use `_replace`.


In [13]:
Employee = namedtuple('Employee', 'id name department salary')

employees = [
    Employee(1, 'Ada', 'Engineering', 140_000),
    Employee(2, 'Grace', 'Engineering', 135_000),
    Employee(3, 'Linus', 'Infrastructure', 150_000),
    Employee(4, 'Guido', 'Engineering', 145_000),
]

employees

[Employee(id=1, name='Ada', department='Engineering', salary=140000),
 Employee(id=2, name='Grace', department='Engineering', salary=135000),
 Employee(id=3, name='Linus', department='Infrastructure', salary=150000),
 Employee(id=4, name='Guido', department='Engineering', salary=145000)]

### Solution

In [14]:
def promote(records, department, raise_percent):
    multiplier = 1 + raise_percent / 100

    promoted = []
    for employee in records:
        if employee.department == department:
            new_salary = round(employee.salary * multiplier, 2)
            promoted.append(employee._replace(salary=new_salary))
        else:
            promoted.append(employee)

    return promoted

new_employees = promote(employees, 'Engineering', 7.5)

for old, new in zip(employees, new_employees):
    print(old, '=>', new, '| same object:', old is new)

Employee(id=1, name='Ada', department='Engineering', salary=140000) => Employee(id=1, name='Ada', department='Engineering', salary=150500.0) | same object: False
Employee(id=2, name='Grace', department='Engineering', salary=135000) => Employee(id=2, name='Grace', department='Engineering', salary=145125.0) | same object: False
Employee(id=3, name='Linus', department='Infrastructure', salary=150000) => Employee(id=3, name='Linus', department='Infrastructure', salary=150000) | same object: True
Employee(id=4, name='Guido', department='Engineering', salary=145000) => Employee(id=4, name='Guido', department='Engineering', salary=155875.0) | same object: False


In [15]:
assert employees[0].salary == 140_000
assert new_employees[0].salary == 150_500
assert new_employees[1].salary == 145_125
assert new_employees[2] is employees[2]
assert new_employees[3].salary == 155_875

### Why unchanged objects are reused

Since the records are immutable, reusing unaffected instances is safe and efficient.


## Problem 6 — Schema extension with default values

Named tuples created with `collections.namedtuple` support field defaults through the `defaults` parameter.

Create an extended `ProductV2` type from `ProductV1` by adding:

- `category`
- `discount`

The default category should be `'general'`, and the default discount should be `0.0`.

Then convert an old `ProductV1` record into a `ProductV2` record using the defaults.


In [16]:
ProductV1 = namedtuple('ProductV1', 'sku name price')

product = ProductV1('BK-001', 'Python Workbook', 39.99)
product

ProductV1(sku='BK-001', name='Python Workbook', price=39.99)

### Solution

In [17]:
ProductV2 = namedtuple(
    'ProductV2',
    ProductV1._fields + ('category', 'discount'),
    defaults=('general', 0.0)
)

def upgrade_product(product, **overrides):
    upgraded = ProductV2(*product)
    return upgraded._replace(**overrides)

default_product = upgrade_product(product)
discounted_product = upgrade_product(product, category='books', discount=0.15)

print(default_product)
print(discounted_product)

assert default_product.category == 'general'
assert default_product.discount == 0.0
assert discounted_product.category == 'books'
assert discounted_product.discount == 0.15

ProductV2(sku='BK-001', name='Python Workbook', price=39.99, category='general', discount=0.0)
ProductV2(sku='BK-001', name='Python Workbook', price=39.99, category='books', discount=0.15)


### Important detail

Defaults apply from the rightmost fields. Because `category` and `discount` were added at the end, they can safely receive defaults without affecting the original required fields.


## Problem 7 — Immutable audit trail

Build a small event-sourcing style update system.

You have an `Account` record and an `AccountEvent` record.

Write `apply_event(account, event)` that supports:

- `'deposit'`
- `'withdraw'`
- `'freeze'`
- `'unfreeze'`

Rules:

1. Frozen accounts cannot deposit or withdraw.
2. Withdrawals cannot make the balance negative.
3. Every successful event increments `version` by 1.
4. Return a new `Account`; never mutate the original.


In [18]:
Account = namedtuple('Account', 'account_id owner balance frozen version')
AccountEvent = namedtuple('AccountEvent', 'type amount')

account = Account('A-100', 'Sofia Ivanova', 500.00, False, 1)
events = [
    AccountEvent('deposit', 250.00),
    AccountEvent('withdraw', 100.00),
    AccountEvent('freeze', 0),
    AccountEvent('unfreeze', 0),
    AccountEvent('withdraw', 50.00),
]

account

Account(account_id='A-100', owner='Sofia Ivanova', balance=500.0, frozen=False, version=1)

### Solution

In [19]:
def apply_event(account, event):
    if event.type in {'deposit', 'withdraw'} and account.frozen:
        raise ValueError('Frozen accounts cannot move funds.')

    if event.type == 'deposit':
        if event.amount <= 0:
            raise ValueError('Deposit amount must be positive.')
        return account._replace(
            balance=round(account.balance + event.amount, 2),
            version=account.version + 1
        )

    if event.type == 'withdraw':
        if event.amount <= 0:
            raise ValueError('Withdrawal amount must be positive.')
        if event.amount > account.balance:
            raise ValueError('Insufficient funds.')
        return account._replace(
            balance=round(account.balance - event.amount, 2),
            version=account.version + 1
        )

    if event.type == 'freeze':
        return account._replace(frozen=True, version=account.version + 1)

    if event.type == 'unfreeze':
        return account._replace(frozen=False, version=account.version + 1)

    raise ValueError(f'Unsupported event type: {event.type!r}')

current = account

for event in events:
    current = apply_event(current, event)
    print(event, '=>', current)

print('Original account:', account)
print('Final account:', current)

AccountEvent(type='deposit', amount=250.0) => Account(account_id='A-100', owner='Sofia Ivanova', balance=750.0, frozen=False, version=2)
AccountEvent(type='withdraw', amount=100.0) => Account(account_id='A-100', owner='Sofia Ivanova', balance=650.0, frozen=False, version=3)
AccountEvent(type='freeze', amount=0) => Account(account_id='A-100', owner='Sofia Ivanova', balance=650.0, frozen=True, version=4)
AccountEvent(type='unfreeze', amount=0) => Account(account_id='A-100', owner='Sofia Ivanova', balance=650.0, frozen=False, version=5)
AccountEvent(type='withdraw', amount=50.0) => Account(account_id='A-100', owner='Sofia Ivanova', balance=600.0, frozen=False, version=6)
Original account: Account(account_id='A-100', owner='Sofia Ivanova', balance=500.0, frozen=False, version=1)
Final account: Account(account_id='A-100', owner='Sofia Ivanova', balance=600.0, frozen=False, version=6)


In [20]:
assert account.balance == 500.00
assert account.version == 1
assert current.balance == 600.00
assert current.frozen is False
assert current.version == 6

### Advanced takeaway

This pattern is powerful when you want value objects: each state transition produces a new immutable state, making debugging and auditability easier.


## Problem 8 — Challenge: generic named tuple extender

Write a function:

```python
extend_namedtuple(base_type, new_type_name, new_fields, defaults=None)
```

It should:

1. Return a new named tuple class.
2. Preserve all fields from `base_type`.
3. Append `new_fields`.
4. Support optional defaults for the new fields only.
5. Reject duplicate field names.

Then use it to create a `Point3D` from `Point2D`.


In [21]:
Point2D = namedtuple('Point2D', 'x y')
point = Point2D(10, 20)
point

Point2D(x=10, y=20)

### Solution

In [22]:
def extend_namedtuple(base_type, new_type_name, new_fields, defaults=None):
    new_fields = tuple(new_fields)
    all_fields = base_type._fields + new_fields

    duplicates = {
        field for field in all_fields
        if all_fields.count(field) > 1
    }

    if duplicates:
        raise ValueError(f'Duplicate field name(s): {sorted(duplicates)}')

    if defaults is not None and len(defaults) > len(new_fields):
        raise ValueError('Defaults may only be supplied for appended fields.')

    return namedtuple(new_type_name, all_fields, defaults=defaults)

Point3D = extend_namedtuple(
    Point2D,
    'Point3D',
    ('z',),
    defaults=(0,)
)

p3_default = Point3D(*point)
p3_custom = Point3D(*point, 30)

print(p3_default)
print(p3_custom)
print(Point3D._fields)

Point3D(x=10, y=20, z=0)
Point3D(x=10, y=20, z=30)
('x', 'y', 'z')


In [23]:
assert p3_default == Point3D(10, 20, 0)
assert p3_custom == Point3D(10, 20, 30)

try:
    extend_namedtuple(Point2D, 'BadPoint', ('x', 'z'))
except ValueError as ex:
    print(type(ex).__name__, ex)

ValueError Duplicate field name(s): ['x']


## Summary

Best practices demonstrated in this notebook:

- Prefer `_replace` for updates to existing named tuple instances.
- Use `_fields` to extend schemas instead of rewriting field lists manually.
- Use `_make` when constructing from an iterable.
- Keep transformations pure: return new values and leave old values unchanged.
- Add validation around named tuple operations when domain rules matter.
- Reuse unchanged immutable objects safely in bulk transformations.
